# Neural Network Implementation from Scratch - Main Results Reproduction

This notebook reproduces the main results from our deep learning project report. It provides a comprehensive overview of the implementation, setup, and key findings.

## Table of Contents

1. [Project Overview](#1-project-overview)
2. [Repository Structure](#2-repository-structure)
3. [Setup and Installation](#3-setup-and-installation)
4. [Data Loading and Preprocessing](#4-data-loading-and-preprocessing)
5. [Model Architecture](#5-model-architecture)
6. [Training Examples](#6-training-examples)
7. [Results Reproduction](#7-results-reproduction)
   - 7.1 [Fashion-MNIST Results](#71-fashion-mnist-results)
   - 7.2 [CIFAR-10 Results](#72-cifar-10-results)
8. [PyTorch Validation](#8-pytorch-validation)
9. [Key Findings](#9-key-findings)
10. [Conclusion](#10-conclusion)

---

## 1. Project Overview

This project implements a complete neural network framework from scratch using only NumPy, without relying on high-level deep learning libraries like TensorFlow or PyTorch. The implementation includes:

- **Forward and backward propagation** with automatic differentiation
- **Multiple activation functions**: ReLU, tanh, sigmoid, softmax
- **Various optimization algorithms**: SGD, Momentum, RMSprop, Adam
- **Regularization techniques**: L2 regularization, dropout
- **Weight initialization schemes**: Xavier/Glorot, He initialization
- **Validation against PyTorch** to ensure correctness

The framework is evaluated on two benchmark datasets:
- **Fashion-MNIST**: 60,000 training and 10,000 test images (28×28 grayscale)
- **CIFAR-10**: 50,000 training and 10,000 test images (32×32 color)

**GitHub Repository**: [https://github.com/[your-username]/DeepLearningGroup71](https://github.com/[your-username]/DeepLearningGroup71)


## 2. Repository Structure

The repository is organized as follows:

```
DeepLearningGroup71/
├── src/                    # Core implementation
│   ├── neural_network.py   # Main NeuralNetwork class
│   ├── layers.py           # DenseLayer implementation
│   ├── activations.py      # Activation functions (ReLU, tanh, sigmoid, softmax)
│   ├── losses.py           # Loss functions (cross-entropy, MSE)
│   ├── optimizers.py       # Optimizers (SGD, Momentum, RMSprop, Adam)
│   ├── initializers.py     # Weight initialization (Xavier, He)
│   ├── data_loader.py      # Data loading utilities
│   └── utils.py            # Utility functions
├── experiments/            # Training scripts
│   ├── train.py           # Main training script with WandB
│   ├── train_simple.py    # Simplified training script
│   └── compare_numpy_pytorch.py  # Validation against PyTorch
├── configs/                # Configuration files
│   ├── default_config.yaml
│   └── fashion_mnist_sweep.yaml
├── data/                   # Dataset storage
├── results/                # Saved models and plots
│   ├── models/            # Trained model checkpoints
│   └── plots/             # Visualization outputs
├── notebooks/              # Jupyter notebooks
├── tests/                  # Unit tests
└── requirements.txt        # Python dependencies
```

### Key Components:

- **`src/neural_network.py`**: Main `NeuralNetwork` class that orchestrates forward/backward passes, training, and evaluation
- **`src/layers.py`**: `DenseLayer` class implementing fully-connected layers
- **`src/activations.py`**: Activation functions and their derivatives
- **`src/optimizers.py`**: Optimization algorithms for weight updates
- **`src/data_loader.py`**: Functions to load and preprocess Fashion-MNIST and CIFAR-10
- **`experiments/train.py`**: Training script with WandB integration for experiment tracking


## 3. Setup and Installation

First, let's set up the environment and import necessary libraries.


In [ ]:
# Import standard libraries
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from pathlib import Path

# Add project root to path
# This notebook is in notebooks/, so project root is parent directory
current_dir = Path().resolve()
if current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    # If running from project root, find notebooks directory and go up
    project_root = current_dir

sys.path.insert(0, str(project_root))

# Import project modules
from src.neural_network import NeuralNetwork
from src.data_loader import (
    load_fashion_mnist, 
    load_cifar10, 
    preprocess_data, 
    create_mini_batches,
    train_val_split,
    get_class_names
)
from src.utils import accuracy_score, set_random_seed

# Set random seed for reproducibility
set_random_seed(42)

print("✓ Imports successful!")
print(f"✓ Project root: {project_root}")
print(f"✓ NumPy version: {np.__version__}")


### Installation Instructions

To set up the environment, install the required packages:

```bash
pip install -r requirements.txt
```

Key dependencies:
- `numpy>=1.26.0` - Core numerical computations
- `matplotlib>=3.8.0` - Visualization
- `wandb>=0.16.0` - Experiment tracking (optional)
- `torch>=2.1.0` - For PyTorch validation (optional)
- `jupyter>=1.0.0` - For running notebooks


## 4. Data Loading and Preprocessing

The project supports two datasets: Fashion-MNIST and CIFAR-10. Let's examine the data loading process.


In [ ]:
# Set data directory
data_dir = project_root / 'data'

# Load Fashion-MNIST dataset
print("Loading Fashion-MNIST dataset...")
X_train_fm, y_train_fm, X_test_fm, y_test_fm = load_fashion_mnist(data_dir)

print(f"\nFashion-MNIST shapes:")
print(f"  Training images: {X_train_fm.shape}")
print(f"  Training labels: {y_train_fm.shape}")
print(f"  Test images: {X_test_fm.shape}")
print(f"  Test labels: {y_test_fm.shape}")

# Load CIFAR-10 dataset
print("\nLoading CIFAR-10 dataset...")
X_train_c10, y_train_c10, X_test_c10, y_test_c10 = load_cifar10(data_dir)

print(f"\nCIFAR-10 shapes:")
print(f"  Training images: {X_train_c10.shape}")
print(f"  Training labels: {y_train_c10.shape}")
print(f"  Test images: {X_test_c10.shape}")
print(f"  Test labels: {y_test_c10.shape}")

# Display class names
fashion_classes = get_class_names('fashion_mnist')
cifar10_classes = get_class_names('cifar10')

print(f"\nFashion-MNIST classes: {fashion_classes}")
print(f"CIFAR-10 classes: {cifar10_classes}")


In [ ]:
# Visualize sample images from both datasets
fig, axes = plt.subplots(2, 5, figsize=(12, 6))

# Fashion-MNIST samples
for i in range(5):
    axes[0, i].imshow(X_train_fm[i], cmap='gray')
    axes[0, i].set_title(f"FM: {fashion_classes[y_train_fm[i]]}")
    axes[0, i].axis('off')

# CIFAR-10 samples
for i in range(5):
    axes[1, i].imshow(X_train_c10[i])
    axes[1, i].set_title(f"C10: {cifar10_classes[y_train_c10[i]]}")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()


### Data Preprocessing

The data preprocessing pipeline includes:
1. **Flattening**: Images are reshaped from 2D/3D to 1D vectors
   - Fashion-MNIST: 28×28 → 784 features
   - CIFAR-10: 32×32×3 → 3072 features
2. **Normalization**: Pixel values scaled to [0, 1] by dividing by 255
3. **One-hot encoding**: Labels converted to one-hot vectors for multi-class classification
4. **Train/Validation split**: 20% of training data used for validation


In [ ]:
# Preprocess Fashion-MNIST
X_train_fm_proc, y_train_fm_proc = preprocess_data(
    X_train_fm, y_train_fm, 
    num_classes=10, 
    flatten=True, 
    normalize=True
)
X_test_fm_proc, y_test_fm_proc = preprocess_data(
    X_test_fm, y_test_fm,
    num_classes=10,
    flatten=True,
    normalize=True
)

# Split into train/val
X_train_fm_final, X_val_fm, y_train_fm_final, y_val_fm = train_val_split(
    X_train_fm_proc, y_train_fm_proc,
    val_split=0.2,
    random_seed=42
)

print("Fashion-MNIST preprocessing complete:")
print(f"  Train: {X_train_fm_final.shape}, {y_train_fm_final.shape}")
print(f"  Val: {X_val_fm.shape}, {y_val_fm.shape}")
print(f"  Test: {X_test_fm_proc.shape}, {y_test_fm_proc.shape}")


In [ ]:
# Preprocess CIFAR-10
X_train_c10_proc, y_train_c10_proc = preprocess_data(
    X_train_c10, y_train_c10,
    num_classes=10,
    flatten=True,
    normalize=True
)
X_test_c10_proc, y_test_c10_proc = preprocess_data(
    X_test_c10, y_test_c10,
    num_classes=10,
    flatten=True,
    normalize=True
)

# Split into train/val
X_train_c10_final, X_val_c10, y_train_c10_final, y_val_c10 = train_val_split(
    X_train_c10_proc, y_train_c10_proc,
    val_split=0.2,
    random_seed=42
)

print("CIFAR-10 preprocessing complete:")
print(f"  Train: {X_train_c10_final.shape}, {y_train_c10_final.shape}")
print(f"  Val: {X_val_c10.shape}, {y_val_c10.shape}")
print(f"  Test: {X_test_c10_proc.shape}, {y_test_c10_proc.shape}")


## 5. Model Architecture

The neural network is implemented as a fully-connected feedforward network with the following components:

### 5.1 Architecture Components

1. **Layers**: Fully-connected (dense) layers
   - Each layer performs: $Z = XW + b$ followed by activation $A = \sigma(Z)$
   
2. **Activation Functions**:
   - **ReLU**: $f(x) = \max(0, x)$ - Used in hidden layers
   - **Tanh**: $f(x) = \tanh(x)$ - Alternative for hidden layers
   - **Sigmoid**: $f(x) = \frac{1}{1+e^{-x}}$ - Alternative for hidden layers
   - **Softmax**: $f(x_c) = \frac{e^{x_c}}{\sum_k e^{x_k}}$ - Used in output layer

3. **Loss Function**: Cross-entropy loss for multi-class classification
   $$\mathcal{L} = -\frac{1}{m} \sum_{i=1}^{m} \sum_{c=1}^{K} y_{i,c} \log(\hat{y}_{i,c})$$

4. **Optimizers**:
   - **SGD**: $W = W - \alpha \nabla_W$
   - **Momentum**: $v = \beta v - \alpha \nabla_W$, $W = W + v$
   - **RMSprop**: Adaptive learning rate per parameter
   - **Adam**: Combines momentum and RMSprop with bias correction

5. **Regularization**:
   - **L2 Regularization**: Adds $\frac{\lambda}{2m} \sum W^2$ to loss
   - **Dropout**: Randomly sets activations to zero during training

6. **Weight Initialization**:
   - **Xavier/Glorot**: For tanh/sigmoid activations
   - **He**: For ReLU activations


In [ ]:
# Example: Create a simple model to demonstrate architecture
example_model = NeuralNetwork(
    input_size=784,  # Fashion-MNIST input size
    hidden_layers=[256, 128],  # Two hidden layers
    output_size=10,  # 10 classes
    activation='relu',
    output_activation='softmax',
    learning_rate=0.001,
    optimizer='adam',
    weight_init='he',
    l2_lambda=0.0001,
    dropout_rate=0.0,
    random_seed=42
)

print("Model architecture:")
print(f"  Input size: {example_model.input_size}")
print(f"  Hidden layers: {example_model.hidden_layers}")
print(f"  Output size: {example_model.output_size}")
print(f"  Activation: {example_model.activation}")
print(f"  Optimizer: {example_model.optimizer_name}")
print(f"  L2 lambda: {example_model.l2_lambda}")

# Display layer information
print("\nLayer details:")
for i, layer in enumerate(example_model.layers):
    print(f"  Layer {i+1}: {layer.W.shape[0]} → {layer.W.shape[1]} ({layer.activation})")


## 6. Training Examples

Let's demonstrate the training process with a quick example on a small subset of data.


In [ ]:
def train_epoch(model, X_train, y_train, batch_size=32):
    """Train for one epoch."""
    batches = create_mini_batches(X_train, y_train, batch_size=batch_size, shuffle=True)
    losses = []
    predictions = []
    labels = []
    
    for X_batch, y_batch in batches:
        loss = model.train_step(X_batch, y_batch)
        losses.append(loss)
        
        preds = model.predict(X_batch)
        predictions.append(preds)
        
        if y_batch.ndim > 1:
            y_batch_indices = np.argmax(y_batch, axis=1)
        else:
            y_batch_indices = y_batch
        labels.append(y_batch_indices)
    
    avg_loss = np.mean(losses)
    all_preds = np.concatenate(predictions)
    all_labels = np.concatenate(labels)
    accuracy = accuracy_score(all_preds, all_labels)
    
    return avg_loss, accuracy

def evaluate(model, X_val, y_val):
    """Evaluate model on validation/test set."""
    model.eval()
    y_pred_proba = model.predict_proba(X_val)
    y_pred = model.predict(X_val)
    
    loss = model.compute_loss(y_pred_proba, y_val)
    
    if y_val.ndim > 1 and y_val.shape[1] > 1:
        y_val_indices = np.argmax(y_val, axis=1)
    else:
        y_val_indices = y_val
    
    accuracy = accuracy_score(y_pred, y_val_indices)
    return loss, accuracy

print("✓ Training utilities defined")


## 7. Results Reproduction

Now let's reproduce the main results from the report. We'll train models with the best configurations found during hyperparameter tuning.


### 7.1 Fashion-MNIST Results

According to the report, the best configuration for Fashion-MNIST achieved **88-90% validation accuracy** using:
- Architecture: [256, 128] hidden layers
- Activation: ReLU with He initialization
- Optimizer: Adam with learning rate 0.001
- L2 regularization: 0.0001
- Batch size: 32, 50 epochs


In [ ]:
# Create model with best Fashion-MNIST configuration
model_fm = NeuralNetwork(
    input_size=784,
    hidden_layers=[256, 128],
    output_size=10,
    activation='relu',
    output_activation='softmax',
    learning_rate=0.001,
    optimizer='adam',
    weight_init='he',
    l2_lambda=0.0001,
    dropout_rate=0.0,
    random_seed=42
)

# Training configuration
num_epochs = 50
batch_size = 32

# Training history
train_losses = []
train_accs = []
val_losses = []
val_accs = []

print("Training Fashion-MNIST model...")
print(f"Configuration: [256, 128] layers, ReLU, Adam, LR=0.001, L2=0.0001")

for epoch in range(num_epochs):
    model_fm.train()
    train_loss, train_acc = train_epoch(model_fm, X_train_fm_final, y_train_fm_final, batch_size)
    
    val_loss, val_acc = evaluate(model_fm, X_val_fm, y_val_fm)
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
              f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

# Final evaluation on test set
test_loss, test_acc = evaluate(model_fm, X_test_fm_proc, y_test_fm_proc)
print(f"\n✓ Training complete!")
print(f"Final Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")


In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(val_losses, label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Fashion-MNIST: Training and Validation Loss', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(train_accs, label='Train Accuracy', linewidth=2)
axes[1].plot(val_accs, label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Fashion-MNIST: Training and Validation Accuracy', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best Validation Accuracy: {max(val_accs):.4f} ({max(val_accs)*100:.2f}%)")
print(f"Final Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")


### 7.2 CIFAR-10 Results

According to the report, the best CIFAR-10 configuration achieved **80.9% validation accuracy** using:
- Architecture: [256] (single hidden layer)
- Activation: Tanh with Xavier initialization
- Optimizer: Adam with learning rate ~0.003
- L2 regularization: ~0.003
- Dropout rate: ~0.1
- Batch size: 32-64

Note: CIFAR-10 is more challenging, and we'll use a simplified configuration for demonstration.


In [ ]:
# Create model with best CIFAR-10 configuration (simplified for faster training)
# Note: Full training may take longer, so we use a smaller configuration for demonstration
model_c10 = NeuralNetwork(
    input_size=3072,  # 32*32*3 = 3072
    hidden_layers=[256],  # Single hidden layer (best performing)
    output_size=10,
    activation='tanh',  # Tanh performed best on CIFAR-10
    output_activation='softmax',
    learning_rate=0.003,
    optimizer='adam',
    weight_init='xavier',  # Xavier for tanh
    l2_lambda=0.003,
    dropout_rate=0.1,
    random_seed=42
)

# Training configuration
num_epochs_c10 = 30  # Reduced for demonstration
batch_size_c10 = 64

# Training history
train_losses_c10 = []
train_accs_c10 = []
val_losses_c10 = []
val_accs_c10 = []

print("Training CIFAR-10 model...")
print(f"Configuration: [256] layer, Tanh, Xavier init, Adam, LR=0.003, L2=0.003, Dropout=0.1")

for epoch in range(num_epochs_c10):
    model_c10.train()
    train_loss, train_acc = train_epoch(model_c10, X_train_c10_final, y_train_c10_final, batch_size_c10)
    
    val_loss, val_acc = evaluate(model_c10, X_val_c10, y_val_c10)
    
    train_losses_c10.append(train_loss)
    train_accs_c10.append(train_acc)
    val_losses_c10.append(val_loss)
    val_accs_c10.append(val_acc)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs_c10}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
              f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

# Final evaluation on test set
test_loss_c10, test_acc_c10 = evaluate(model_c10, X_test_c10_proc, y_test_c10_proc)
print(f"\n✓ Training complete!")
print(f"Final Test Accuracy: {test_acc_c10:.4f} ({test_acc_c10*100:.2f}%)")


In [ ]:
# Plot training curves for CIFAR-10
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(train_losses_c10, label='Train Loss', linewidth=2)
axes[0].plot(val_losses_c10, label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('CIFAR-10: Training and Validation Loss', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(train_accs_c10, label='Train Accuracy', linewidth=2)
axes[1].plot(val_accs_c10, label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('CIFAR-10: Training and Validation Accuracy', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best Validation Accuracy: {max(val_accs_c10):.4f} ({max(val_accs_c10)*100:.2f}%)")
print(f"Final Test Accuracy: {test_acc_c10:.4f} ({test_acc_c10*100:.2f}%)")


## 8. PyTorch Validation

To ensure correctness, the NumPy implementation was validated against PyTorch by comparing:
1. Forward pass outputs
2. Loss values
3. Gradients after backward pass
4. Weight updates after training steps

Results showed numerical agreement within acceptable tolerances (relative tolerance $10^{-3}$).

Let's demonstrate a quick validation:


In [ ]:
# Quick validation example (requires PyTorch)
try:
    import torch
    from src.pytorch_neural_network import PyTorchNeuralNetwork
    
    # Create small test models
    test_input_size = 784
    test_hidden = [64]
    test_output = 10
    
    numpy_model = NeuralNetwork(
        input_size=test_input_size,
        hidden_layers=test_hidden,
        output_size=test_output,
        activation='relu',
        learning_rate=0.001,
        optimizer='adam',
        weight_init='he',
        random_seed=42
    )
    
    pytorch_model = PyTorchNeuralNetwork(
        input_size=test_input_size,
        hidden_layers=test_hidden,
        output_size=test_output,
        activation='relu',
        learning_rate=0.001,
        optimizer='adam',
        weight_init='he',
        random_seed=42
    )
    
    # Copy weights from NumPy to PyTorch
    numpy_params = numpy_model.get_params()
    pytorch_model.set_params(numpy_params)
    
    # Test on small batch
    X_test_small = X_train_fm_final[:32]
    y_test_small = y_train_fm_final[:32]
    
    # Forward pass comparison
    numpy_model.eval()
    y_numpy = numpy_model.predict_proba(X_test_small)
    
    X_torch = torch.from_numpy(X_test_small).float()
    pytorch_model.eval()
    y_pytorch = pytorch_model.predict_proba(X_torch).cpu().numpy()
    
    # Compare outputs
    max_diff = np.max(np.abs(y_numpy - y_pytorch))
    mean_diff = np.mean(np.abs(y_numpy - y_pytorch))
    is_close = np.allclose(y_numpy, y_pytorch, rtol=1e-3, atol=1e-5)
    
    print("PyTorch Validation Results:")
    print(f"  Max difference: {max_diff:.2e}")
    print(f"  Mean difference: {mean_diff:.2e}")
    print(f"  Matches (rtol=1e-3): {is_close}")
    
    if is_close:
        print("✓ NumPy and PyTorch implementations match!")
    else:
        print("⚠ Small differences detected (may be due to numerical precision)")
        
except ImportError:
    print("PyTorch not available. Skipping validation.")
    print("To run validation, install PyTorch: pip install torch")
except Exception as e:
    print(f"Validation error: {e}")
    print("This is expected if PyTorch models are not available.")


## 9. Key Findings

Based on the experiments and results, here are the key findings:

### 9.1 Activation Functions
- **Tanh outperformed ReLU on CIFAR-10**, while **ReLU worked better on Fashion-MNIST**
- This highlights dataset-specific optimization needs

### 9.2 Architecture Depth
- **Deeper networks did not always improve performance**
- Single-layer [256] achieved best CIFAR-10 results
- Suggests importance of capacity vs. regularization balance

### 9.3 Optimization
- **Adam provided more stable convergence** than SGD
- SGD with proper learning rate scheduling performed comparably for well-tuned hyperparameters

### 9.4 Regularization
- **L2 regularization and dropout were crucial** for preventing overfitting
- Optimal L2 values were lower than initially expected (0.002-0.005)
- Dropout rates of 0.05-0.15 were optimal

### 9.5 Weight Initialization
- **Xavier initialization worked best with tanh**
- **He initialization was optimal for ReLU**
- Confirms theoretical expectations

### 9.6 Performance Summary

| Dataset | Best Architecture | Activation | Test Accuracy |
|---------|------------------|------------|---------------|
| Fashion-MNIST | [256, 128] | ReLU + He | 88-90% |
| CIFAR-10 | [256] | Tanh + Xavier | 65-81% (varies by config) |


## 10. Conclusion

This project successfully implemented a complete neural network framework from scratch using NumPy, demonstrating:

1. **Deep understanding** of core deep learning concepts including forward/backward propagation, optimization algorithms, and regularization techniques

2. **Correctness validation** through systematic comparison with PyTorch, achieving numerical agreement within acceptable tolerances

3. **Competitive performance** on benchmark datasets:
   - Fashion-MNIST: 88-90% accuracy
   - CIFAR-10: 65-81% accuracy (depending on configuration)

4. **Comprehensive hyperparameter tuning** using WandB sweeps on HPC infrastructure

5. **Insights into dataset-specific optimization strategies**

### Key Contributions

- Complete, modular neural network implementation with multiple optimizers and regularization techniques
- Systematic hyperparameter tuning using WandB sweeps
- Validation of correctness through comparison with PyTorch
- Insights into dataset-specific optimization strategies

### Limitations and Future Work

The current implementation uses fully-connected layers, which are computationally expensive for image data. Future improvements could include:

- Convolutional layers for better image feature extraction
- Batch normalization for improved training stability
- Learning rate scheduling for better convergence
- Data augmentation to improve generalization

---

## Repository Information

**GitHub Repository**: [https://github.com/[your-username]/DeepLearningGroup71](https://github.com/[your-username]/DeepLearningGroup71)

**Project Structure**: See Section 2 for detailed repository organization.

**Main Training Script**: `experiments/train.py` - Full training script with WandB integration

**Validation Script**: `experiments/compare_numpy_pytorch.py` - PyTorch comparison script

---

*This notebook reproduces the main results from the project report. For full hyperparameter sweeps and detailed experiments, refer to the WandB project dashboard or run the training scripts directly.*
